In [6]:
!pip install mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.2/131.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.5/838.5 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 14.5 MB/s eta 0:00:00


In [2]:
import pandas as pd

train_df = pd.read_csv("https://raw.githubusercontent.com/MadExplorer/MLops/main/Practice_3/Data/train.csv")
test_df = pd.read_csv("https://raw.githubusercontent.com/MadExplorer/MLops/main/Practice_3/Data/test.csv")

print(train_df.shape)
print(test_df.shape)
print(train_df.head())

(1460, 81)
(1459, 80)
   Id  MSSubClass MSZoning  LotFrontage  LotArea Street Alley LotShape  \
0   1          60       RL         65.0     8450   Pave   NaN      Reg   
1   2          20       RL         80.0     9600   Pave   NaN      Reg   
2   3          60       RL         68.0    11250   Pave   NaN      IR1   
3   4          70       RL         60.0     9550   Pave   NaN      IR1   
4   5          60       RL         84.0    14260   Pave   NaN      IR1   

  LandContour Utilities  ... PoolArea PoolQC Fence MiscFeature MiscVal MoSold  \
0         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
1         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      5   
2         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      9   
3         Lvl    AllPub  ...        0    NaN   NaN         NaN       0      2   
4         Lvl    AllPub  ...        0    NaN   NaN         NaN       0     12   

  YrSold  SaleType  SaleCondition  SalePrice  

In [3]:
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

target_col = "SalePrice"

X = train_df.drop(columns=[target_col])
y = train_df[target_col]

# Убираем Id из признаков
if "Id" in X.columns:
    X = X.drop(columns=["Id"])

X_test_final = test_df.copy()
if "Id" in X_test_final.columns:
    X_test_final = X_test_final.drop(columns=["Id"])

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [4]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

def eval_regression(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

In [8]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
from mlflow.models import infer_signature
import mlflow

ridge_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

ridge_params = {
    "model__alpha": [0.1, 1.0, 10.0, 100.0]
}

with mlflow.start_run(run_name="Ridge_Regression"):
    grid = GridSearchCV(
        ridge_pipeline,
        ridge_params,
        cv=5,
        scoring="r2",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    preds = best_model.predict(X_val)

    rmse, mae, r2 = eval_regression(y_val, preds)

    mlflow.log_param("model", "Ridge")
    mlflow.log_param("alpha", grid.best_params_["model__alpha"])

    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("r2", float(r2))

    signature = infer_signature(X_train, best_model.predict(X_train))
    mlflow.sklearn.log_model(best_model, artifact_path="model", signature=signature)

    print("Ridge best params:", grid.best_params_)
    print(f"RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.4f}")

2026/04/05 07:50:55 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/04/05 07:50:55 INFO mlflow.store.db.utils: Updating database tables
/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/05 07:51:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Plea

Ridge best params: {'model__alpha': 10.0}
RMSE=30646.31, MAE=19040.56, R2=0.8776


In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

rf_params = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [10, 20, None],
    "model__min_samples_split": [2, 5]
}

with mlflow.start_run(run_name="RandomForest_Regressor"):
    grid = GridSearchCV(
        rf_pipeline,
        rf_params,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    preds = best_model.predict(X_val)

    rmse, mae, r2 = eval_regression(y_val, preds)

    mlflow.log_param("model", "RandomForestRegressor")
    mlflow.log_param("n_estimators", grid.best_params_["model__n_estimators"])
    mlflow.log_param("max_depth", grid.best_params_["model__max_depth"])
    mlflow.log_param("min_samples_split", grid.best_params_["model__min_samples_split"])

    mlflow.log_metric("rmse", float(rmse))
    mlflow.log_metric("mae", float(mae))
    mlflow.log_metric("r2", float(r2))

    signature = infer_signature(X_train, best_model.predict(X_train))
    mlflow.sklearn.log_model(best_model, artifact_path="model", signature=signature)

    print("RandomForest best params:", grid.best_params_)
    print(f"RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.4f}")

/usr/local/lib/python3.12/dist-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/05 07:56:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/05 07:56:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution becaus

RandomForest best params: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 100}
RMSE=28928.53, MAE=17654.98, R2=0.8909


In [10]:
runs = mlflow.search_runs()
runs_sorted = runs.sort_values("metrics.r2", ascending=False)

runs_sorted[[
    "run_id",
    "params.model",
    "params.alpha",
    "params.n_estimators",
    "params.max_depth",
    "params.min_samples_split",
    "metrics.rmse",
    "metrics.mae",
    "metrics.r2"
]]

,run_id,params.model,params.alpha,params.n_estimators,params.max_depth,params.min_samples_split,metrics.rmse,metrics.mae,metrics.r2
0,3b9b52b85b4d40aa89f64975a281eadf,RandomForestRegressor,None,100,None,5,28928.534419,17654.980636,0.890896
1,c3d3e0638b004184969d66baf76eb760,Ridge,10.0,None,None,None,30646.308031,19040.559638,0.877555


In [11]:
best_run = runs.sort_values("metrics.r2", ascending=False).iloc[0]

best_run_id = best_run["run_id"]
model_uri = f"runs:/{best_run_id}/model"

print("Best run id:", best_run_id)
print("Model URI:", model_uri)
print(best_run[["params.model", "metrics.rmse", "metrics.mae", "metrics.r2"]])

Best run id: 3b9b52b85b4d40aa89f64975a281eadf
Model URI: runs:/3b9b52b85b4d40aa89f64975a281eadf/model
params.model    RandomForestRegressor
metrics.rmse             28928.534419
metrics.mae              17654.980636
metrics.r2                   0.890896
Name: 0, dtype: object


In [12]:
loaded_model = mlflow.sklearn.load_model(model_uri)
loaded_model

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'YearBuilt',
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                                   '1stFlrSF', '2ndFlrSF',
                                                   'LowQual...
                                                   'Neighborhood', 'Condition1',
                                                   'Condition2', 'BldgType',
                                                   'HouseStyle', 'RoofStyle',
                                                   'RoofMatl', 'Exterior1st',
                                                   'Exterior2nd', 'MasVnrType',
                                                   'ExterQual', 'ExterCond',
                                                   'Foundation', 'BsmtQual',
                                                   'BsmtCond', 'BsmtExposure',
                                                   'BsmtFinType1',
                                                   'BsmtFinType2', 'Heating',
                                                   'HeatingQC', 'CentralAir',
                                                   'Electrical', ...])])),
                ('model',
                 RandomForestRegressor(min_samples_split=5, n_jobs=-1,
                                       random_state=42))])